In [15]:
from __future__ import annotations

import json
from datetime import timedelta
from io import BytesIO
from typing import Any, BinaryIO, Dict, Iterable, Optional
import socket

import urllib3
from urllib3.util import Timeout

from loguru import logger
from minio import Minio
from minio.error import S3Error
from pydantic import BaseModel, Field

class MinioSettings(BaseModel):
    port: str
    host: str
    user:str
    password: str
    access_key: str = Field(default="minioadmin", description="MinIO access key")
    secret_key: str = Field(default="minioadmin", description="MinIO secret key")
    secure: bool = Field(default=False, description="Whether to use HTTPS when contacting MinIO")

setting = MinioSettings(
    port="9000",
    user='minioadmin',
    host='localhost',
    password='minioadmin',
    access_key='minioadmin',
    secret_key='minioadmin',
    secure=False
)


In [16]:
class StorageClient:
    def __init__(self, settings:  MinioSettings) -> None:
        self.settings = settings 
        timeout = Timeout(connect=5.0, read=120.0)  
        self._http_client = urllib3.PoolManager(
            maxsize=50,
            timeout=timeout,
            block=True,
        )
        self.client = Minio(
            endpoint=f"{settings.host}:{settings.port}",
            access_key=settings.access_key,
            secret_key=settings.secret_key,
            secure=settings.secure,
            http_client=self._http_client,
        )

    def _ensure_bucket(self, bucket: str) -> None:
        try:
            if not self.client.bucket_exists(bucket):
                print(f"Bucket named: {bucket} does not exist, creating")
                self.client.make_bucket(bucket)
        except S3Error as exc:
            print("MinIO bucket check failed for %s: %s", bucket, exc)



    def upload_fileobj(
        self,
        bucket: str,
        object_name: str,
        file_obj: BinaryIO,
        *,
        content_type: str = "application/octet-stream",
        metadata: Optional[Dict[str, str]] = None,
    ) -> str:
        """Upload a file-like object to the specified bucket and return an s3 URI."""

        self._ensure_bucket(bucket)
        try:
            self.client.put_object(
                bucket_name=bucket,
                object_name=object_name,
                data=file_obj,
                length=-1,
                part_size=10 * 1024 * 1024,
                content_type=content_type,
                metadata=metadata, #type:ignore
            )
            uri = f"s3://{bucket}/{object_name}"
            print("Uploaded object %s", uri)
            return uri
        except S3Error as exc:
            print("Failed to upload %s to bucket %s", object_name, bucket)

    def put_json(
        self,
        bucket: str,
        object_name: str,
        payload: Dict[str, Any],
        *,
        content_type: str = "application/json",
    ) -> str:
        buffer = BytesIO(json.dumps(payload, separators=(",", ":")).encode("utf-8"))
        return self.upload_fileobj(
            bucket,
            object_name,
            buffer,
            content_type=content_type,
            metadata={"Content-Type": content_type},
        )
    def get_presigned_url(
        self,
        bucket: str,
        object_name: str,
        *,
        expires_seconds: timedelta = timedelta(seconds=3600),
    ) -> str:
        self._ensure_bucket(bucket)
        try:
            url = self.client.presigned_get_object(bucket, object_name, expires=expires_seconds)
            print("Generated presigned URL for %s/%s", bucket, object_name)
            return url
        except S3Error as exc:
            print("Failed to generate presigned URL for %s/%s", bucket, object_name)

    def list_objects(self, bucket: str, prefix: Optional[str] = None) -> Iterable[str]:
        self._ensure_bucket(bucket)
        try:
            for obj in self.client.list_objects(bucket, prefix=prefix or "", recursive=True):
                if obj.object_name is not None:
                    yield obj.object_name
        except S3Error as exc:
            print("Failed to list objects for %s/%s", bucket, prefix)

    def get_object(self, bucket: str, object_name: str) -> bytes | None:
        self._ensure_bucket(bucket)
        try:
            response = self.client.get_object(bucket, object_name)
            try:
                data = response.read()
                return data
            finally:
                response.close()
                response.release_conn()
        except S3Error as exc:
            print(f"Bucket: {bucket} has no {object_name}")
            return None
    def read_json(self, bucket: str, object_name: str) -> Dict[str, Any] | None:
        raw = self.get_object(bucket, object_name)
        if raw is None:
            return None
        try:
            return json.loads(raw.decode("utf-8"))
        except json.JSONDecodeError as exc:
            raise ValueError(f"Stored object {bucket}/{object_name} is not valid JSON: {exc}") from exc

    def object_exists(self, bucket:str, object_name: str) -> bool:
        self._ensure_bucket(bucket)
        try:
            self.client.stat_object(bucket, object_name)
            return True
        except S3Error as exc:
            if exc.code in ("NoSuchKey", "NoSuchObject"):
                return False
            raise ValueError(f"Error checking object {bucket}/{object_name}: {exc}") from exc

In [17]:
minio_class = StorageClient(settings=setting)

In [22]:
caption_check = ["caption/image/video1_111", "caption/image/video2_222", "caption/image/video3_333"]

from collections import defaultdict

video2caption = defaultdict(list)


for video_name in caption_check:

    objects = minio_class.list_objects('anonymous', video_name)
    for obj in objects:
        json_content = minio_class.read_json(bucket='anonymous', object_name=obj)
        video2caption[video_name].append(json_content)


In [23]:
video2caption[caption_check[1]]

[{'artifact_type': 'ImageArtifact',
  'frame_index': 28,
  'time_stamp': '00:00:00.933',
  'related_video_id': 'video2_222',
  'related_video_fps': 30.0,
  'extension': '.webp',
  'user_bucket': 'anonymous',
  'image_minio_url': 's3://anonymous/images/video2_222/00000028_00:00:00.933.webp',
  'image_id': 'c8726ba5d9e252cd2d26783d5190f0cdf7d3ab54d2081f59b76b75a51c7bf9bc7185c0ff4d8adb015ed9c24bef40a7d12253d22c9513cfc7a6e737862bda8989',
  'caption': 'Bức ảnh này là một cảnh quay đồ họa, có lẽ là phần mở đầu (intro) hoặc màn hình chuyển cảnh của một chương trình tin tức truyền hình mang tên **"TIN TỨC 24H"** của kênh **"Tin24h"** (Logo hình tròn màu đỏ nằm ở góc trên bên trái).\n\n**Bố cục tổng thể** được thiết kế theo phong cách năng động, hiện đại với tông màu chủ đạo là xanh dương đậm và trắng, sử dụng nhiều hiệu ứng đồ họa 3D và các ô hình ảnh nhỏ chia cắt bố cục.\n\n**Trung tâm của màn hình** là một dải băng màu xanh đậm nổi bật với dòng chữ trắng in hoa lớn: **"TIN TỨC 24H"**. Phía s

In [27]:
segment_caption_check = ["caption/segment/video1_111", "caption/segment/video2_222", "caption/segment/video3_333"]


video2segment_caption = defaultdict(list)


for video_name in segment_caption_check:

    objects = minio_class.list_objects('anonymous', video_name)
    for obj in objects:
        json_content = minio_class.read_json(bucket='anonymous', object_name=obj)
        video2segment_caption[video_name].append(json_content)


In [29]:
video2segment_caption[segment_caption_check[1]]

[{'artifact_type': 'SegmentCaptionArtifact',
  'autoshot_artifact_id': 'd6f787463c698bd5ee40c4e76f36ae38da08062817087a465c34feedef32d62d46f151d748c29eebefec5d916914244d0534049dd5d0032d12ca2be464e70dc2',
  'related_video_extension': 'mp4',
  'related_video_id': 'video2_222',
  'related_video_fps': 30.0,
  'start_frame': 0,
  'end_frame': 112,
  'start_timestamp': '00:00:00.000',
  'end_timestamp': '00:00:03.733',
  'related_asr': '',
  'related_video_minio_url': 's3://testbucket/videoplayback_2.mp4',
  'user_bucket': 'anonymous',
  'caption': 'Đoạn video mở đầu bằng đồ họa chuyển động nền màu xanh dương với hình ảnh quả địa cầu xoay tròn ở trung tâm, trên đó có dòng chữ lớn màu trắng nổi bật: "**TIN TỨC 24H**". Xung quanh là các ô hình ảnh ghép lại, thể hiện nhiều sự kiện thời sự khác nhau, bao gồm: một cơ sở công nghiệp đang xả khói, cựu Tổng thống Mỹ Donald Trump đang ký tài liệu, Tổng thống Nga Vladimir Putin đang phát biểu, một máy bay chiến đấu đậu trên đường băng và một vụ phóng t